# Octospawn

Autorzy: **Kalina Raczka, Emilia Sarna**

## Opis gry

### Wersja deterministyczna
Octospawn to wariant Hexapawn na planszy **4x4**.
Każdy gracz startuje z 4 pionkami (gracz 1 na górze, gracz 2 na dole).
Ruchy: pion o jedno pole do przodu lub bicie po skosie o jedno pole.
Gracz przegrywa, gdy: przeciwnik dojdzie pionkiem do linii celu lub nie ma żadnego legalnego ruchu.

### Wersja probabilistyczna
Po wykonaniu ruchu wystepuje **10% szansy** na odrodzenie jednego z uprzednio zbitych pionkow
aktualnego gracza. Pionek jest wybierany losowo z listy zbitych i wraca na swoje pole startowe.


## Zastosowane algorytmy
Porownano 3 warianty:
- Negamax bez odciecia alfa-beta
- Negamax z odcieciem alfa-beta
- ExpectiMiniMax z odcieciem alfa-beta (tylko dla prob=0.1)

### Negamax

In [ ]:
# mynegamax.py
if bestValue < move_alpha:
    bestValue = move_alpha
    best_move = move
    if depth == origDepth:
        state.ai_move = move


### Negamax z odcieciem alfa-beta
W tym wariancie aktywne są granice `alpha` i `beta`, dzięki czemu część gałęzi
drzewa nie jest przeglądana. Zmniejsza to czas wyboru ruchu przy zachowaniu dobrego wyniku.

In [ ]:
if alpha < move_alpha and pruning:
    alpha = move_alpha
    if alpha >= beta:
        break

### ExpectiMiniMax
Implementacja modeluje wezly losowe (szansa odrodzenia się pionka) i liczy wartość oczekiwaną ruchu.
W eksperymentach uzyto tylko wersji z odcieciem alfa-beta.

In [ ]:
# expectiminimax.py
move_alpha_no = -expectiminimax(game, depth - 1, origDepth, scoring, -beta, -alpha, pruning=pruning)

if moving_player_has_lost_pawns:
    move_alpha_yes = -expectiminimax(game_with_forced_resurrection, depth - 1, origDepth, scoring, -beta, -alpha, pruning=pruning)
    move_alpha = (1 - game.chance) * move_alpha_no + game.chance * move_alpha_yes
else:
    move_alpha = move_alpha_no

if bestValue < move_alpha:
    bestValue = move_alpha
    if depth == origDepth:
        state.ai_move = move

## Eksperymenty

### Protokol
- glebokosci: **A=3** oraz **B=4**
- liczba gier na matchup: **40**
- warianty planszy: `chance=0.0` (deterministyczny) i `chance=0.1` (probabilistyczny)
- gracz rozpoczynajacy zmieniany naprzemiennie w kolejnych grach
- metryki: wygrane/przegrane/remisy, sredni czas wyboru ruchu, srednia dlugosc gry

### Wsparcie implementacyjne w main.py
Skrypt wyswietla plansze po kazdym ruchu i podsumowanie kazdej gry (tryb `OCTOSPAWN_TRACE=1`),
a nastepnie drukuje czytelne tabele statystyk koncowych.

Do zebrania danych do tabel uruchomiono tryb bez sladu ruch-po-ruchu:
`OCTOSPAWN_TRACE=0 OCTOSPAWN_GAMES=40 python3 main.py`.

In [ ]:
# main.py - kluczowe fragmenty odpowiedzialne za czytelnosc i statystyki
TRACE_EACH_MOVE = env_bool('OCTOSPAWN_TRACE', True)

if trace_each_move:
    print('Initial board:')
    game.show()

move = game.get_move()
game.play_move(move)
if trace_each_move:
    print(f'Move {moves_played}: player {current_player_before} -> {move}')
    game.show()

print_results_table(deterministic, 'Deterministic variant (chance=0.0)')
print_results_table(probabilistic, 'Probabilistic variant (chance=0.1)')

## Wyniki

### 1) Wariant deterministyczny (`chance=0.0`)
| Algorytm | Gry | Wygrane A (d=3) | Wygrane B (d=4) | Remisy | Sredni czas A [s] | Sredni czas B [s] | Srednia dlugosc gry |
|---|---:|---:|---:|---:|---:|---:|---:|
| Negamax_noAB | 40 | 20 | 20 | 0 | 0.002937 | 0.010247 | 10.00 |
| Negamax_AB | 40 | 20 | 20 | 0 | 0.001213 | 0.002162 | 10.00 |
| ExpectiMiniMax_AB | 40 | 20 | 20 | 0 | 0.003851 | 0.009507 | 10.00 |

### 2) Wariant probabilistyczny (`chance=0.1`)
| Algorytm | Gry | Wygrane A (d=3) | Wygrane B (d=4) | Remisy | Sredni czas A [s] | Sredni czas B [s] | Srednia dlugosc gry |
|---|---:|---:|---:|---:|---:|---:|---:|
| Negamax_noAB | 40 | 18 | 22 | 0 | 0.002934 | 0.010406 | 9.65 |
| Negamax_AB | 40 | 23 | 17 | 0 | 0.001161 | 0.002085 | 9.97 |
| ExpectiMiniMax_AB | 40 | 19 | 21 | 0 | 0.002765 | 0.007324 | 9.78 |

### Wnioski
1. Odciecie alfa-beta wyraznie przyspiesza obliczenia (zarowno dla Negamax, jak i ExpectiMiniMax).
2. W wariancie deterministycznym wyniki 3 vs 4 sa symetryczne po wyrownaniu startujacego gracza.
3. W wariancie probabilistycznym pojawia sie roznice skutecznosci miedzy metodami.
4. ExpectiMiniMax lepiej odzwierciedla charakter gry z losowoscia, bo jawnie modeluje wezly szansy.

### Napotkane problemy i ich rozwiazania
- Problem: bledna propagacja parametrow alfa-beta w ExpectiMiniMax.
  Rozwiazanie: poprawiono kolejnosc i nazewnictwo argumentow (`alpha`, `beta`, `pruning`).
- Problem: zly wybor ruchu korzeniowego przy `pruning=False`.
  Rozwiazanie: aktualizacja `state.ai_move` przy kazdej poprawie najlepszego wyniku.
- Problem: identyfikacja gracza po `switch_player()` dla logiki odrodzenia.
  Rozwiazanie: uzycie poprawnej strony (`game.opponent` jako gracz, ktory wykonal ruch).

In [ ]:
# Dane liczbowe (kopiowalne do dalszej analizy / wykresow)
results = {
    'deterministic': [
        {'variant': 'Negamax_noAB', 'games': 40, 'A_wins': 20, 'B_wins': 20, 'draws': 0, 'A_time': 0.002937, 'B_time': 0.010247, 'avg_len': 10.00},
        {'variant': 'Negamax_AB', 'games': 40, 'A_wins': 20, 'B_wins': 20, 'draws': 0, 'A_time': 0.001213, 'B_time': 0.002162, 'avg_len': 10.00},
        {'variant': 'ExpectiMiniMax_AB', 'games': 40, 'A_wins': 20, 'B_wins': 20, 'draws': 0, 'A_time': 0.003851, 'B_time': 0.009507, 'avg_len': 10.00}
    ],
    'probabilistic': [
        {'variant': 'Negamax_noAB', 'games': 40, 'A_wins': 18, 'B_wins': 22, 'draws': 0, 'A_time': 0.002934, 'B_time': 0.010406, 'avg_len': 9.65},
        {'variant': 'Negamax_AB', 'games': 40, 'A_wins': 23, 'B_wins': 17, 'draws': 0, 'A_time': 0.001161, 'B_time': 0.002085, 'avg_len': 9.97},
        {'variant': 'ExpectiMiniMax_AB', 'games': 40, 'A_wins': 19, 'B_wins': 21, 'draws': 0, 'A_time': 0.002765, 'B_time': 0.007324, 'avg_len': 9.78}
    ]
}